In [1]:
# ==============================
# 1. IMPORTS
# ==============================
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer
)
from peft import LoraConfig, get_peft_model


g:\v1\env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ==============================
# 2. CONFIG
# ==============================
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
data_path = "finalV1.json"   # your dataset file


In [3]:
# ==============================
# 3. LOAD DATASET
# ==============================
dataset = load_dataset("json", data_files=data_path)["train"]

In [4]:
# ==============================
# 4. FORMAT PROMPT
# ==============================
def format_example(example):
    replies = "\n".join(example["output"])

    text = (
        "### Instruction:\n"
        f"{example['instruction']}\n\n"
        "### Conversation:\n"
        f"{example['input']}\n\n"
        "### Response:\n"
        f"{replies}"
    )

    return {"text": text}
dataset = dataset.map(format_example)

In [5]:
# ==============================
# 5. TOKENIZER
# ==============================
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token


g:\v1\env\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [6]:
# ==============================
# 6. TOKENIZATION + LABELS (FIX)
# ==============================
def tokenize(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

    # ✅ IMPORTANT: add labels for loss
    tokens["labels"] = tokens["input_ids"].copy()

    return tokens

# ==============================
# SPLIT DATASET
# ==============================
dataset = dataset.train_test_split(test_size=0.1)

train_dataset = dataset["train"]
eval_dataset = dataset["test"]

# ==============================
# TOKENIZE AFTER SPLIT ✅
# ==============================
train_dataset = train_dataset.map(tokenize, batched=True)
eval_dataset = eval_dataset.map(tokenize, batched=True)

# remove unused columns
train_dataset = train_dataset.remove_columns(["instruction", "input", "output", "text"])
eval_dataset = eval_dataset.remove_columns(["instruction", "input", "output", "text"])

Map: 100%|██████████| 567/567 [00:00<00:00, 4428.93 examples/s]


In [7]:

# ==============================
# 7. LOAD MODEL (4-bit QLoRA)
# ==============================
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)
from peft import prepare_model_for_kbit_training

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

# ✅ VERY IMPORTANT (FIXES YOUR ERROR)
model = prepare_model_for_kbit_training(model)




In [8]:
# ==============================
# 8. APPLY LoRA
# ==============================
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)


model.print_trainable_parameters()

trainable params: 2,252,800 || all params: 1,102,301,184 || trainable%: 0.20437245579516677


In [9]:
# ==============================
# 9. TRAINING CONFIG
# ==============================
training_args = TrainingArguments(
    output_dir="./model",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=2,
    learning_rate=1e-4,
    fp16=True,

    logging_steps=10,
    logging_strategy="steps",

    save_strategy="epoch",
    save_total_limit=2,

    evaluation_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    report_to="none"
)

In [10]:
# ==============================
# 10. TRAINER
# ==============================
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer
)


g:\v1\env\lib\site-packages\accelerate\accelerator.py:450: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


In [11]:
# ==============================
# 11. TRAIN 🚀
# ==============================
trainer.train()


  0%|          | 0/1274 [00:00<?, ?it/s]`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
g:\v1\env\lib\site-packages\torch\_dynamo\eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
  1%|          | 10/1274 [00:18<36:37,  1.74s/it]

{'loss': 12.3608, 'grad_norm': 36.47510528564453, 'learning_rate': 9.937205651491366e-05, 'epoch': 0.02}


  2%|▏         | 20/1274 [00:35<35:23,  1.69s/it]

{'loss': 4.8837, 'grad_norm': 19.90633201599121, 'learning_rate': 9.858712715855574e-05, 'epoch': 0.03}


  2%|▏         | 30/1274 [00:54<40:07,  1.94s/it]

...

                                                   
100%|██████████| 1274/1274 [37:11<00:00,  1.64s/it]

{'eval_loss': 0.117413230240345, 'eval_runtime': 22.0575, 'eval_samples_per_second': 25.706, 'eval_steps_per_second': 3.219, 'epoch': 2.0}


100%|██████████| 1274/1274 [37:12<00:00,  1.75s/it]

{'train_runtime': 2232.1311, 'train_samples_per_second': 4.571, 'train_steps_per_second': 0.571, 'train_loss': 0.2788925157798516, 'epoch': 2.0}


TrainOutput(global_step=1274, training_loss=0.2788925157798516, metrics={'train_runtime': 2232.1311, 'train_samples_per_second': 4.571, 'train_steps_per_second': 0.571, 'train_loss': 0.2788925157798516, 'epoch': 2.0})

In [13]:
# ==============================
# 12. SAVE MODEL
# ==============================
model.save_pretrained("newsmart_reply_model")
tokenizer.save_pretrained("newsmart_reply_model")

print("✅ Training Complete!")

✅ Training Complete!
